# Chapter 22: Dynamic Attributes and Properties

**Fluent Python, 2nd Edition** -- Luciano Ramalho

> "The crucial importance of properties is that their existence makes it perfectly safe and indeed advisable for you to expose public data attributes as part of your class's public interface."
> -- Martelli, Ravenscroft, and Holden

This notebook covers:
- `__getattr__` / `__setattr__` for dynamic attribute access
- The `@property` decorator and the `property()` constructor
- Property factories for DRY validation
- Caching computed properties with `functools`
- Attribute deletion
- Built-in functions: `vars`, `getattr`, `hasattr`, `setattr`, `dir`


---
## 1. Dynamic Attributes with `__getattr__`

`__getattr__` is called **only** when normal attribute lookup fails (i.e. the name is not found in the instance `__dict__`, nor in the class or superclasses). This makes it ideal for building flexible wrappers.


In [ ]:
from collections import abc

class FrozenJSON:
    """Read-only facade for navigating JSON-like data with dot notation."""

    def __init__(self, mapping):
        self.__data = dict(mapping)

    def __getattr__(self, name):
        try:
            return getattr(self.__data, name)  # delegate dict methods (.keys, .items, ...)
        except AttributeError:
            return FrozenJSON.build(self.__data[name])  # recurse into nested data

    def __dir__(self):
        return self.__data.keys()

    @classmethod
    def build(cls, obj):
        if isinstance(obj, abc.Mapping):
            return cls(obj)
        elif isinstance(obj, abc.MutableSequence):
            return [cls.build(item) for item in obj]
        else:
            return obj

# ── Demo ──
data = {
    "Schedule": {
        "events": [{"serial": 100, "name": "My Talk", "speakers": [1, 2]}],
        "speakers": [{"serial": 1, "name": "Alice"}, {"serial": 2, "name": "Bob"}],
    }
}

feed = FrozenJSON(data)
print("Keys:", sorted(feed.Schedule.keys()))
print("First event:", feed.Schedule.events[0].name)
print("Speakers:", [s.name for s in feed.Schedule.speakers])


### How `__getattr__` Works



Key point: `__getattr__` is a **fallback**, not called on every access. Contrast with `__getattribute__`, which is called on *every* attribute access.


### Using `__new__` for Flexible Object Creation

`__init__` is an initializer, not a constructor. The real constructor is `__new__`. It can return instances of **different types**, and when it does, `__init__` is skipped.


In [ ]:
import keyword
from collections import abc

class FrozenJSON2:
    """FrozenJSON using __new__ instead of a build() classmethod."""

    def __new__(cls, arg):
        if isinstance(arg, abc.Mapping):
            return super().__new__(cls)  # normal: create a FrozenJSON2 instance
        elif isinstance(arg, abc.MutableSequence):
            return [cls(item) for item in arg]  # return a list, __init__ is NOT called
        else:
            return arg  # return the scalar as-is

    def __init__(self, mapping):
        self.__data = {}
        for key, value in mapping.items():
            if keyword.iskeyword(key):
                key += "_"  # e.g. "class" -> "class_"
            self.__data[key] = value

    def __getattr__(self, name):
        try:
            return getattr(self.__data, name)
        except AttributeError:
            return FrozenJSON2(self.__data[name])

# ── Demo with keyword handling ──
student = FrozenJSON2({"name": "Jim Bo", "class": 1982})
print("Name:", student.name)
print("Class (note trailing _):", student.class_)

# __new__ returning non-FrozenJSON2 types:
result = FrozenJSON2(42)
print(f"FrozenJSON2(42) -> {result!r} (type: {type(result).__name__})")
result2 = FrozenJSON2([1, 2, 3])
print(f"FrozenJSON2([1,2,3]) -> {result2!r}")


---
## 2. Computed Properties with `@property`

The `@property` decorator turns a method into a **read-only computed attribute**. This lets you add computation behind what looks like simple attribute access, following the Uniform Access Principle.


In [ ]:
import json

class Record:
    """Simple record built from keyword arguments (the "bunch" idiom)."""
    __index = None  # class-level cache

    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)  # quick way to set many attributes at once

    def __repr__(self):
        cls = self.__class__.__name__
        serial = getattr(self, "serial", "?")
        return f"<{cls} serial={serial!r}>"

    @staticmethod
    def fetch(key):
        if Record._Record__index is None:
            Record._Record__index = load_data()
        return Record._Record__index[key]


class Event(Record):
    def __repr__(self):
        try:
            return f"<{self.__class__.__name__} {self.name!r}>"
        except AttributeError:
            return super().__repr__()

    @property
    def venue(self):
        """Dereference venue_serial -> Record."""
        key = f"venue.{self.venue_serial}"
        return self.__class__.fetch(key)

    @property
    def speakers(self):
        """Dereference speaker serial numbers -> list of Records."""
        spkr_serials = self.__dict__["speakers"]  # bypass property to avoid recursion!
        fetch = self.__class__.fetch
        return [fetch(f"speaker.{key}") for key in spkr_serials]


def load_data():
    """Build a flat index of Record/Event objects from sample data."""
    raw_data = {
        "Schedule": {
            "conferences": [{"serial": 1}],
            "events": [
                {"serial": 33950, "name": "There *Will* Be Bugs",
                 "event_type": "40-minute session", "venue_serial": 1449,
                 "speakers": [3471, 5199]},
            ],
            "speakers": [
                {"serial": 3471, "name": "Anna Ravenscroft", "twitter": "annaraven"},
                {"serial": 5199, "name": "Alex Martelli", "twitter": "aleax"},
            ],
            "venues": [
                {"serial": 1449, "name": "Portland 251"},
            ],
        }
    }
    import inspect
    records = {}
    for collection, raw_records in raw_data["Schedule"].items():
        record_type = collection[:-1]
        cls_name = record_type.capitalize()
        # Look up Event (or any future subclass) by name
        cls = {"Event": Event}.get(cls_name, Record)
        if inspect.isclass(cls) and issubclass(cls, Record):
            factory = cls
        else:
            factory = Record
        for raw_record in raw_records:
            key = f"{record_type}.{raw_record['serial']}"
            records[key] = factory(**raw_record)
    return records


# ── Demo ──
event = Record.fetch("event.33950")
print(event)
print("Venue:", event.venue, "->", event.venue.name)
print("Speakers:", [(s.serial, s.name) for s in event.speakers])


### Why `self.__dict__["speakers"]` in the Property?

The `speakers` property has the same name as the JSON field stored in the instance. If we wrote `self.speakers` inside the property method, Python would call the property again, causing infinite recursion.

**Properties are overriding descriptors** -- they always shadow instance attributes of the same name. Reading from `self.__dict__` directly bypasses the property.


---
## 3. Caching Properties

Three approaches for caching computed property results:
1. **`functools.cached_property`** -- stores result as same-named instance attr (non-overriding descriptor)
2. **`@property` + `@functools.cache`** -- works when name clashes with existing attr
3. **Hand-rolled cache** -- legacy approach, beware of PEP 412 key-sharing issues


In [ ]:
from functools import cached_property, cache

class Circle:
    def __init__(self, radius):
        self.radius = radius

    # 1. cached_property -- ideal for new computed attributes
    @cached_property
    def area(self):
        print("  [computing area...]")
        import math
        return math.pi * self.radius ** 2

c = Circle(5)
print("First access:")
print(f"  area = {c.area:.4f}")
print("Second access (cached, no recomputation):")
print(f"  area = {c.area:.4f}")
print()

# You can clear the cache by deleting the attribute
del c.area
print("After del c.area, third access:")
print(f"  area = {c.area:.4f}")


In [ ]:
from functools import cache

class EventLike:
    """Demonstrates @property + @cache for name-clashing attributes."""
    def __init__(self, speaker_ids):
        self.__dict__["speakers"] = speaker_ids  # raw data stored here

    # @property on top of @cache  (order matters!)
    @property
    @cache
    def speakers(self):
        raw = self.__dict__["speakers"]
        print(f"  [resolving {raw}...]")
        return [f"Speaker-{sid}" for sid in raw]

e = EventLike([3471, 5199])
print("First:", e.speakers)
print("Second (cached):", e.speakers)


### `cached_property` Limitations

- Cannot replace `@property` when the method depends on an **instance attribute with the same name** (because `cached_property` is a non-overriding descriptor).
- Cannot be used in classes with `__slots__`.
- Creates instance attributes *after* `__init__`, defeating PEP 412 key-sharing optimization.
- **Is thread-safe** (uses a reentrant lock internally).


---
## 4. Using Properties for Attribute Validation

A read/write property lets you add validation (e.g., reject negative values) to what was once a plain public attribute -- without changing the class interface.


In [ ]:
class LineItem:
    """A line item in an order, with validated weight and price."""

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight   # triggers the property setter
        self.price = price

    def subtotal(self):
        return self.weight * self.price

    @property
    def weight(self):
        return self.__weight   # stored in private name-mangled attr

    @weight.setter
    def weight(self, value):
        if value > 0:
            self.__weight = value
        else:
            raise ValueError("weight must be > 0")

    @property
    def price(self):
        return self.__price

    @price.setter
    def price(self, value):
        if value > 0:
            self.__price = value
        else:
            raise ValueError("price must be > 0")


# ── Demo ──
raisins = LineItem("Golden raisins", 10, 6.95)
print(f"{raisins.description}: {raisins.subtotal()}")

# Validation in action
try:
    bad = LineItem("walnuts", -5, 10.00)
except ValueError as e:
    print(f"Blocked: {e}")

# Setting after creation also validates
try:
    raisins.weight = -20
except ValueError as e:
    print(f"Blocked: {e}")


### The `property()` Constructor

`property` is actually a class, not a function. Full signature:



The decorator syntax:


is equivalent to:



---
## 5. Property Factories

When the same validation logic applies to multiple attributes, a **property factory** eliminates repetition by returning a `property()` object whose getter/setter closures capture the storage name.


In [ ]:
def quantity(storage_name):
    """Property factory: rejects zero or negative values."""

    def qty_getter(instance):
        return instance.__dict__[storage_name]

    def qty_setter(instance, value):
        if value > 0:
            instance.__dict__[storage_name] = value
        else:
            raise ValueError("value must be > 0")

    return property(qty_getter, qty_setter)


class LineItem2:
    weight = quantity("weight")   # factory builds a property with closure over "weight"
    price  = quantity("price")    # ... and another over "price"

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price


# ── Demo ──
nutmeg = LineItem2("Moluccan nutmeg", 8, 13.95)
print(f"subtotal: {nutmeg.subtotal()}")
print(f"instance __dict__: {nutmeg.__dict__}")

try:
    nutmeg.price = -1
except ValueError as e:
    print(f"Blocked: {e}")


### How the Factory Avoids Infinite Recursion

Inside `qty_getter` and `qty_setter`, we read/write `instance.__dict__[storage_name]` directly instead of using `instance.weight` / `instance.price`. Using the attribute name would trigger the property again, leading to infinite recursion.

The `storage_name` is captured in the closure of each getter/setter. Each call to `quantity()` creates a fresh pair of closures sharing the same `storage_name`.


---
## 6. Properties Override Instance Attributes

A crucial rule: **properties defined on a class always take precedence over instance attributes of the same name**. This is because properties are *overriding descriptors*.


In [ ]:
class Demo:
    data = "class data attr"

    @property
    def prop(self):
        return "the prop value"

obj = Demo()

# Instance attribute shadows class DATA attribute
obj.data = "instance data"
print("obj.data:", obj.data)        # -> "instance data"  (instance wins)
print("Demo.data:", Demo.data)     # -> "class data attr"  (class untouched)

# But property CANNOT be shadowed by instance attribute
print("obj.prop:", obj.prop)        # -> "the prop value" (property wins)

try:
    obj.prop = "override attempt"    # no setter defined -> error
except AttributeError as e:
    print(f"Cannot set: {e}")

# Even writing directly to __dict__ doesn't help:
obj.__dict__["prop"] = "forced"
print("obj.__dict__:", obj.__dict__)
print("obj.prop:", obj.prop)         # STILL calls the property getter!

# The only way to remove the property is to replace it on the CLASS:
Demo.prop = "no longer a property"
print("obj.prop:", obj.prop)         # NOW reads from __dict__: "forced"


---
## 7. Handling Attribute Deletion

The `@my_property.deleter` decorator (or `fdel` argument to `property()`) defines behavior for `del obj.attr`. The lower-level `__delattr__` special method can also intercept deletion.


In [ ]:
class BlackKnight:
    """Inspired by Monty Python and the Holy Grail."""

    def __init__(self):
        self.phrases = [
            ("an arm", "'Tis but a scratch."),
            ("another arm", "It's just a flesh wound."),
            ("a leg", "I'm invincible!"),
            ("another leg", "All right, we'll call it a draw."),
        ]

    @property
    def member(self):
        if self.phrases:
            return self.phrases[0][0]
        return "nothing left!"

    @member.deleter
    def member(self):
        if self.phrases:
            member, text = self.phrases.pop(0)
            print(f"BLACK KNIGHT (loses {member}) -- {text}")

knight = BlackKnight()
print("Next member:", knight.member)

del knight.member
del knight.member
del knight.member
del knight.member
print("Next member:", knight.member)

---
## 8. `__setattr__` -- Intercepting All Attribute Writes

Unlike `__getattr__` (fallback only), `__setattr__` is called on **every** attribute assignment. This makes it powerful but also tricky -- you must avoid infinite recursion.


In [ ]:
class Validated:
    """Demonstrates __setattr__ for universal validation."""
    _valid_attrs = {"name", "age", "email"}

    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)  # goes through __setattr__

    def __setattr__(self, name, value):
        if name not in self._valid_attrs:
            raise AttributeError(f"Cannot set unknown attribute: {name!r}")
        if name == "age" and not isinstance(value, int):
            raise TypeError(f"age must be int, got {type(value).__name__}")
        # Use super().__setattr__ to actually store it (avoids recursion)
        super().__setattr__(name, value)

    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"Validated({attrs})"

v = Validated(name="Alice", age=30, email="alice@example.com")
print(v)

try:
    v.phone = "555-1234"
except AttributeError as e:
    print(f"Blocked: {e}")

try:
    v.age = "thirty"
except TypeError as e:
    print(f"Blocked: {e}")


---
## 9. Essential Built-in Functions for Attribute Handling

| Function | Purpose |
|---|---|
| `dir(obj)` | List "interesting" attributes (triggers `__dir__`) |
| `getattr(obj, name[, default])` | Fetch attribute, with optional fallback |
| `hasattr(obj, name)` | `True` if `getattr` doesn't raise `AttributeError` |
| `setattr(obj, name, value)` | Assign attribute (triggers `__setattr__`) |
| `vars(obj)` | Return `obj.__dict__` (fails with `__slots__`-only classes) |


In [ ]:
class Widget:
    kind = "generic"
    __slots__ = ()  # no __dict__

class FancyWidget:
    kind = "fancy"
    def __init__(self, color="red"):
        self.color = color
    def __dir__(self):
        return ["color", "kind", "paint"]  # custom dir output

w = FancyWidget("blue")

# vars() returns __dict__
print("vars(w):", vars(w))

# getattr with fallback
print("getattr(w, 'size', 'N/A'):", getattr(w, "size", "N/A"))

# hasattr checks existence
print("hasattr(w, 'color'):", hasattr(w, "color"))
print("hasattr(w, 'size'):", hasattr(w, "size"))

# dir() uses __dir__ if defined
print("dir(w):", dir(w))

# setattr
setattr(w, "style", "modern")
print("After setattr:", vars(w))


---
## 10. Special Attributes for Attribute Handling

| Attribute | Description |
|---|---|
| `__class__` | Reference to the object's class (`type(obj)`) |
| `__dict__` | Mapping of writable attributes for the object/class |
| `__slots__` | Tuple of allowed attribute names (saves memory, no `__dict__`) |

### Special Methods Summary

| Method | When Called |
|---|---|
| `__getattr__(self, name)` | Fallback -- only when normal lookup fails |
| `__getattribute__(self, name)` | **Every** attribute access (use with extreme care) |
| `__setattr__(self, name, value)` | **Every** attribute assignment |
| `__delattr__(self, name)` | `del obj.attr` |
| `__dir__(self)` | `dir(obj)` -- customize attribute listing |


In [ ]:
class MyObj:
    class_var = 42
    def __init__(self, x):
        self.x = x

obj = MyObj(10)
print("__class__:", obj.__class__)
print("type(obj):", type(obj))
print("obj.__dict__:", obj.__dict__)
print("MyObj.__dict__ keys:", list(MyObj.__dict__.keys()))

# __class__ and type() are the same
assert obj.__class__ is type(obj)

# __dict__ is where instance attributes live
obj.__dict__["y"] = 99
print("After direct __dict__ write, obj.y =", obj.y)


---
## Key Takeaways

1. **`__getattr__`** is a fallback for missing attributes; **`__getattribute__`** intercepts every access.
2. **`__setattr__`** is always called on assignment. Use `super().__setattr__` or `self.__dict__` to store values.
3. **`@property`** creates overriding descriptors that shadow same-named instance attributes.
4. **`functools.cached_property`** is a non-overriding descriptor -- simpler but cannot shadow existing attrs.
5. **Property factories** use closures to generate reusable property objects for DRY validation.
6. Access `instance.__dict__` directly to bypass properties and avoid recursion.
7. The `property()` constructor accepts `fget`, `fset`, `fdel`, and `doc`.
8. `vars()`, `getattr()`, `hasattr()`, `setattr()`, and `dir()` are the essential built-in functions for attribute introspection.
